In [2]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# one way anova
# 1. Create the dataset based on the lecture slides
data_weight = {
    'Weight_Loss': [
        8, 9, 6, 7, 3,  # Low Calorie
        2, 4, 3, 5, 1,  # Low Fat
        3, 5, 4, 2, 3,  # Low Carbohydrate
        2, 2, -1, 0, 3  # Control
    ],
    'Diet_Group': [
        'Low_Calorie']*5 + ['Low_Fat']*5 + ['Low_Carb']*5 + ['Control']*5
}

df_weight = pd.DataFrame(data_weight)

# 2. Fit the Ordinary Least Squares (OLS) model
# The formula 'Weight_Loss ~ C(Diet_Group)' tells the model that Diet_Group is a categorical factor
model_weight = ols('Weight_Loss ~ C(Diet_Group)', data=df_weight).fit()

# 3. Generate the ANOVA table
anova_table_weight = sm.stats.anova_lm(model_weight, typ=2)

print("--- One-Way ANOVA: Weight Loss ---")
print(anova_table_weight)
print("\n")

--- One-Way ANOVA: Weight Loss ---
               sum_sq    df         F    PR(>F)
C(Diet_Group)   75.75   3.0  8.559322  0.001278
Residual        47.20  16.0       NaN       NaN




In [3]:
# Two way anova

# 1. Create the dataset based on the lecture slides
data_sales = {
    'Sales': [47, 43, 46, 40, 62, 68, 67, 71, 41, 39, 42, 46],
    'Height': [1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3],
    'Width': [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
}

df_sales = pd.DataFrame(data_sales)

# 2. Fit the OLS model
# The formula includes Height, Width, and their interaction (Height:Width)
# C() is used to ensure the model treats numerical inputs (1, 2, 3) as distinct categories
model_sales = ols('Sales ~ C(Height) + C(Width) + C(Height):C(Width)', data=df_sales).fit()

# 3. Generate the ANOVA table
anova_table_sales = sm.stats.anova_lm(model_sales, typ=2)

print("--- Two-Way ANOVA: Bread Sales ---")
print(anova_table_sales)

--- Two-Way ANOVA: Bread Sales ---
                    sum_sq   df          F    PR(>F)
C(Height)           1544.0  2.0  74.709677  0.000058
C(Width)              12.0  1.0   1.161290  0.322605
C(Height):C(Width)    24.0  2.0   1.161290  0.374697
Residual              62.0  6.0        NaN       NaN


In [1]:
import numpy as np
from scipy.stats import f

# -------------------------
# Data (from the table)
# -------------------------
t1 = np.array([8, 9, 6, 8, 5], dtype=float)
t2 = np.array([5, 4, 7, 6, 6], dtype=float)
t3 = np.array([9, 3, 2, 4], dtype=float)

groups = [t1, t2, t3]
k = len(groups)
n_i = np.array([len(g) for g in groups])
N = n_i.sum()

# -------------------------
# Means
# -------------------------
means = np.array([g.mean() for g in groups])
grand_mean = np.concatenate(groups).mean()

# -------------------------
# Sums of Squares
# -------------------------
SS_between = np.sum(n_i * (means - grand_mean) ** 2)
SS_within = np.sum([np.sum((g - g.mean()) ** 2) for g in groups])
SS_total = np.sum((np.concatenate(groups) - grand_mean) ** 2)

# -------------------------
# ANOVA table quantities
# -------------------------
df_between = k - 1
df_within = N - k

MS_between = SS_between / df_between
MS_within = SS_within / df_within

F_stat = MS_between / MS_within
p_value = f.sf(F_stat, df_between, df_within)  # right-tail p-value

alpha = 0.10
decision = "Reject H0" if p_value < alpha else "Fail to reject H0"

# -------------------------
# Print results
# -------------------------
print("Group sizes (n_i):", n_i)
print("Group means:", means)
print("Grand mean:", grand_mean)

print("\nSS_between:", SS_between)
print("SS_within :", SS_within)
print("SS_total  :", SS_total)

print("\ndf_between:", df_between, " df_within:", df_within)
print("MS_between:", MS_between)
print("MS_within :", MS_within)

print("\nF statistic:", F_stat)
print("p-value   :", p_value)

print(f"\nAt alpha = {alpha}: {decision}")

Group sizes (n_i): [5 5 4]
Group means: [7.2 5.6 4.5]
Grand mean: 5.857142857142857

SS_between: 16.714285714285715
SS_within : 45.0
SS_total  : 61.714285714285715

df_between: 2  df_within: 11
MS_between: 8.357142857142858
MS_within : 4.090909090909091

F statistic: 2.042857142857143
p-value   : 0.17601409150114572

At alpha = 0.1: Fail to reject H0


In [3]:
import pandas as pd
import numpy as np

# Load dataset (update column names if needed)
df = pd.read_csv("ut_regression_hw6_data.csv")

# Convert to categorical (important for grouping)
df["program"] = df["program"].astype("category")
df["gender"] = df["gender"].astype("category")

# Basic counts
a = df["program"].nunique()
b = df["gender"].nunique()
N = len(df)

grand_mean = df["grade"].mean()

# Means
mean_A = df.groupby("program")["grade"].mean()
mean_B = df.groupby("gender")["grade"].mean()
mean_AB = df.groupby(["program", "gender"])["grade"].mean()

# Cell sizes
n_A = df.groupby("program").size()
n_B = df.groupby("gender").size()
n_AB = df.groupby(["program", "gender"]).size()

# -------------------------
# SSA (Factor A: Program)
# -------------------------
SSA = sum(n_A[i] * (mean_A[i] - grand_mean)**2 for i in mean_A.index)

# -------------------------
# SSB (Factor B: Gender)
# -------------------------
SSB = sum(n_B[j] * (mean_B[j] - grand_mean)**2 for j in mean_B.index)

# -------------------------
# SSAB (Interaction)
# -------------------------
SSAB = 0
for (i, j) in mean_AB.index:
    SSAB += n_AB[(i, j)] * (
        mean_AB[(i, j)]
        - mean_A[i]
        - mean_B[j]
        + grand_mean
    )**2

print("SSA (Program):", SSA)
print("SSB (Gender):", SSB)
print("SSAB (Interaction):", SSAB)

SSA (Program): 233.24449999999894
SSB (Gender): 38.92050000000057
SSAB (Interaction): 5317.060500000005


In [4]:
# ---------- SSB ----------
mean_B = df.groupby("gender")["grade"].mean()
n_B = df.groupby("gender").size()

SSB = sum(n_B[j] * (mean_B[j] - grand_mean)**2 for j in mean_B.index)

df_B = len(mean_B) - 1
MSB = SSB / df_B

# ---------- SSE ----------
# Compute cell means
mean_AB = df.groupby(["program", "gender"])["grade"].mean()

SSE = 0
for idx, row in df.iterrows():
    cell_mean = mean_AB[(row["program"], row["gender"])]
    SSE += (row["grade"] - cell_mean) ** 2

df_E = len(df) - len(mean_AB)
MSE = SSE / df_E

print("SSB:", SSB)
print("MSB:", MSB)

print("SSE:", SSE)
print("MSE:", MSE)

SSB: 38.92050000000057
MSB: 38.92050000000057
SSE: 1419.214
MSE: 18.67386842105263


In [6]:
grand_mean = df["grade"].mean()

# Counts
a = df["program"].nunique()
b = df["gender"].nunique()
N = len(df)

# Means
mean_A = df.groupby("program")["grade"].mean()
mean_B = df.groupby("gender")["grade"].mean()
mean_AB = df.groupby(["program", "gender"])["grade"].mean()

n_A = df.groupby("program").size()
n_AB = df.groupby(["program", "gender"]).size()

# -------------------------
# SSA
# -------------------------
SSA = sum(n_A[i] * (mean_A[i] - grand_mean)**2 for i in mean_A.index)
df_A = a - 1
MSA = SSA / df_A

# -------------------------
# SSAB
# -------------------------
SSAB = 0
for (i, j) in mean_AB.index:
    SSAB += n_AB[(i, j)] * (
        mean_AB[(i, j)]
        - mean_A[i]
        - mean_B[j]
        + grand_mean
    )**2

df_AB = (a - 1) * (b - 1)
MSAB = SSAB / df_AB

# -------------------------
# SSE and MSE
# -------------------------
SSE = 0
for idx, row in df.iterrows():
    cell_mean = mean_AB[(row["program"], row["gender"])]
    SSE += (row["grade"] - cell_mean)**2

df_E = N - a*b
MSE = SSE / df_E

# -------------------------
# F statistics
# -------------------------
F_A = MSA / MSE
F_B = MSB / MSE
F_AB = MSAB / MSE

print("F_A (Program):", F_A)
print("F_AB (Interaction):", F_AB)

F_A (Program): 12.490422163253688
F_AB (Interaction): 284.7326745649355


In [7]:
from scipy.stats import f
# ---------- p-values ----------
p_A = f.sf(F_A, df_A, df_E)
p_B = f.sf(F_B, df_B, df_E)

print("F_A:", F_A)
print("p_A:", p_A)

print("F_B:", F_B)
print("p_B:", p_B)

F_A: 12.490422163253688
p_A: 0.0006999534520863779
F_B: 2.0842226753682276
p_B: 0.15293759875467375
